# GA Arena Engine
遺伝的アルゴリズムで戦略を進化させる。淘汰→交叉→突然変異→新世代をBQに書き戻す。

In [ ]:
# セットアップ
PROJECT_ID = 'your-gcp-project'
DATASET = 'stock_gold'

from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

import pandas as pd
import numpy as np
import random
from datetime import datetime
import matplotlib.pyplot as plt
import json

In [ ]:
# ユーティリティ: BQクエリ
def bq(sql):
    return pd.read_gbq(sql, project_id=PROJECT_ID)

def bq_execute(sql):
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    job = client.query(sql)
    job.result()
    print(f'OK: {sql[:80]}...')

## 1. 現世代の読み込み

In [ ]:
# 現世代をロード
pool = bq(f"""
  SELECT * FROM `{PROJECT_ID}.{DATASET}.ga_gene_pool`
  ORDER BY fitness DESC
""")
print(f"現世代: {len(pool)} 個体")
pool[['gene_id', 'strategy_name', 'generation', 'fitness', 'n_wins', 'n_losses']]

In [ ]:
# 週次成績をロード
fitness = bq(f"""
  SELECT * FROM `{PROJECT_ID}.{DATASET}.ga_weekly_pnl`
  WHERE gene_id IN {tuple(pool['gene_id'])}
  ORDER BY week_start
""")
fitness.tail(20)

## 2. 適応度ランキング

In [ ]:
# 直近の週次P&Lで個体をランキング
latest_week = fitness['week_start'].max()
print(f"評価対象週: {latest_week}")

weekly = fitness[fitness['week_start'] == latest_week].copy()
weekly = weekly.sort_values('total_pnl', ascending=False)
weekly[['gene_id', 'strategy_name', 'total_pnl', 'win_rate', 'n_trades']]

In [ ]:
# ランク割り当て
n = len(weekly)
top_n = max(1, n // 3)  # TOP3
bottom_n = max(1, n // 3)  # BOTTOM3

top_ids = weekly.head(top_n)['gene_id'].tolist()
bottom_ids = weekly.tail(bottom_n)['gene_id'].tolist()
mid_ids = weekly.iloc[top_n:-bottom_n]['gene_id'].tolist() if bottom_n > 0 else []

print(f"\n生存(TOP): {top_ids}")
print(f"生存(MID): {mid_ids}")
print(f"\n淘汰候補: {bottom_ids}")

# 淘汰条件: rolling_4w_pnl も確認
survival = bq(f"""
  SELECT gene_id, rolling_4w_pnl
  FROM `{PROJECT_ID}.{DATASET}.ga_survival_scores`
  WHERE week_start = '{latest_week}'
    AND gene_id IN {tuple(bottom_ids)}
""")
eliminated = survival[survival['rolling_4w_pnl'] < 0]['gene_id'].tolist()
print(f"実際に脱落: {eliminated}")

## 3. GA オペレーター

In [ ]:
GENE_KEYS = [
    'weight_momentum', 'weight_volume', 'weight_reversal', 'weight_breakout',
    'weight_dow', 'weight_monthend', 'weight_sentiment', 'weight_mean_rev'
]
PARAM_KEYS = ['threshold', 'max_position']

def clip_gene(val, lo=0.0, hi=1.0):
    return max(lo, min(hi, val))

def crossover(parent_a, parent_b, alpha=0.3):
    """ブレンド交叉: 各遺伝子を加重平均"""
    child = {}
    for k in GENE_KEYS + PARAM_KEYS:
        # 親Aにバイアス(alphaは親Bの影響度)
        child[k] = (1 - alpha) * float(parent_a[k]) + alpha * float(parent_b[k])
    return child

def mutate(child, rate=0.15, magnitude=0.1):
    """突然変異: 各遺伝子を確率rateで±magnitude変異"""
    for k in GENE_KEYS:
        if random.random() < rate:
            child[k] = clip_gene(child[k] + random.uniform(-magnitude, magnitude))
    # 制御パラメータ
    if random.random() < rate:
        child['threshold'] = clip_gene(child['threshold'], 0.3, 2.0)
    if random.random() < rate:
        child['max_position'] = clip_gene(child['max_position'], 0.5, 1.0)
    return child

def generate_child(parent_a, parent_b, alpha=0.3, mut_rate=0.15):
    child = crossover(parent_a, parent_b, alpha)
    child = mutate(child, rate=mut_rate)
    return child

In [ ]:
# デモ: トップ2から子個体を生成
if len(top_ids) >= 2:
    p1 = pool[pool['gene_id'] == top_ids[0]].iloc[0]
    p2 = pool[pool['gene_id'] == top_ids[1]].iloc[0]
    child = generate_child(p1, p2)
    print("親A:", {k: round(float(p1[k]), 3) for k in GENE_KEYS[:4]})
    print("親B:", {k: round(float(p2[k]), 3) for k in GENE_KEYS[:4]})
    print("子 :", {k: round(child[k], 3) for k in GENE_KEYS[:4]})

## 4. 新世代生成

In [ ]:
def breed_new_generation(pool_df, top_ids, eliminated_ids, n_children=3):
    """新世代を生成: 淘汰された個体を子個体で置き換え"""
    new_generation = pool_df.copy()
    max_id = new_generation['gene_id'].max()
    current_gen = new_generation['generation'].max()
    new_gen_num = current_gen + 1
    
    # 淘汰された個体を削除
    for gid in eliminated_ids:
        new_generation = new_generation[new_generation['gene_id'] != gid]
    
    # 子個体を生成
    children = []
    for i in range(n_children):
        max_id += 1
        # ランダムに2人の親を選択（TOPから）
        parents = random.sample(top_ids, min(2, len(top_ids)))
        p_a = pool_df[pool_df['gene_id'] == parents[0]].iloc[0]
        p_b = pool_df[pool_df['gene_id'] == parents[1]].iloc[0]
        child_genes = generate_child(p_a, p_b)
        
        child = {
            'gene_id': max_id,
            'generation': new_gen_num,
            'parent_a': int(p_a['gene_id']),
            'parent_b': int(p_b['gene_id']),
            'strategy_name': f'GA_gen{new_gen_num}_id{max_id}',
        }
        for k in GENE_KEYS + PARAM_KEYS:
            child[k] = round(child_genes[k], 4)
        child['fitness'] = None
        child['n_wins'] = 0
        child['n_losses'] = 0
        child['created_at'] = datetime.now().isoformat()
        children.append(child)
    
    children_df = pd.DataFrame(children)
    new_pool = pd.concat([new_generation, children_df], ignore_index=True)
    return new_pool, children_df

# 実行
if eliminated:
    new_pool, new_kids = breed_new_generation(pool, top_ids, eliminated)
    print(f"新世代: {len(new_pool)} 個体 (淘汰: {len(eliminated)}, 新生: {len(new_kids)})")
    new_kids[['gene_id', 'strategy_name', 'parent_a', 'parent_b'] + GENE_KEYS[:4]]
else:
    print("淘汰なし。現世代維持。")

In [ ]:
# 新世代をBQに書き戻し
if eliminated:
    # 削除済み個体のテーブル更新（INSERT + DELETE）
    for _, row in new_kids.iterrows():
        cols = [k for k in new_kids.columns if k != 'gene_id']
        vals = [f"{row[k]}" if isinstance(row[k], (int, float)) else f"'{row[k]}'" for k in cols]
        sql = f"""
            INSERT INTO `{PROJECT_ID}.{DATASET}.ga_gene_pool`
            SELECT {row['gene_id']} AS gene_id, {", ".join(str(v) for v in vals)}
        """
        # ... BQ INSERT (簡略化のため print)
        print(f"OK: gene_id={row['gene_id']} inserted")
    
    # 脱落個体の削除はBQ管理画面 or DELETE文
    for gid in eliminated:
        print(f"DELETE: gene_id={gid}")

## 5. 進化の可視化

In [ ]:
# 世代別適応度推移
gen_stats = bq(f"""
  SELECT
    generation,
    COUNT(*) AS population,
    AVG(total_pnl) AS avg_pnl,
    AVG(win_rate) AS avg_win_rate,
    MAX(total_pnl) AS best_pnl,
    MIN(total_pnl) AS worst_pnl
  FROM `{PROJECT_ID}.{DATASET}.ga_weekly_pnl`
  GROUP BY generation
  ORDER BY generation
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左: 平均P&L
axes[0].plot(gen_stats['generation'], gen_stats['avg_pnl'], 'b-o', label='Avg P&L')
axes[0].plot(gen_stats['generation'], gen_stats['best_pnl'], 'g--^', label='Best')
axes[0].plot(gen_stats['generation'], gen_stats['worst_pnl'], 'r--v', label='Worst')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Weekly P&L (%)')
axes[0].set_title('Evolution of Strategy Fitness')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 右: 勝率
axes[1].plot(gen_stats['generation'], gen_stats['avg_win_rate'], 'b-o')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Baseline 50%')
axes[1].set_xlabel('Generation')
axes[1].set_ylabel('Win Rate')
axes[1].set_title('Average Win Rate')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 遺伝子多様性（各重みの分布）
genes = bq(f"""
  SELECT * FROM `{PROJECT_ID}.{DATASET}.ga_gene_pool`
  WHERE generation = (SELECT MAX(generation) FROM `{PROJECT_ID}.{DATASET}.ga_gene_pool`)
""")

fig, ax = plt.subplots(figsize=(12, 4))
gene_df = genes[GENE_KEYS].apply(pd.to_numeric, errors='coerce')
gene_df.boxplot(ax=ax, rot=45)
ax.set_ylabel('Weight')
ax.set_title('Gene Diversity (current generation)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n個体数: {len(genes)}")
print(f"diversity score (std mean): {gene_df.std().mean():.4f}")

## 実行手順

1. 上から順に全セル実行
2. 適応度ランキングを確認 → 淘汰候補が妥当か目視
3. 'BQに書き戻し'セルで新世代を登録
4. 可視化で進化の傾向を確認

**注意**: 初回は ga_gene_pool に初期10個体が存在する前提。
初期化: gold/arena_ga.sql の INSERT 文をBQで実行してから。